# Modular n-gram language models

This notebook trains and compares unigram, bigram, trigram, and quadrigram language models using one reusable implementation.

In [34]:
from collections import Counter
from math import exp, log
from pathlib import Path

import pandas as pd

# Locate the tokenized corpus whether the notebook runs from Lab 4 or the repository root.
candidates = [
    Path.cwd() / "Lab 1" / "outputs" / "tokenized_sentences_with_source.txt",
    Path.cwd().parent / "Lab 1" / "outputs" / "tokenized_sentences_with_source.txt",
    Path(r"E:\STUDY\NLP\Lab 1\outputs\tokenized_sentences_with_source.txt"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find tokenized_sentences_with_source.txt")

input_sentences = pd.read_json(data_path, lines=True)
sentences = input_sentences["word_tokens"].reset_index(drop=True).tolist()
sentences = [list(sentence) for sentence in sentences if sentence]

if len(sentences) < 3_000:
    raise ValueError("The corpus must contain at least 3,000 non-empty sentences.")

    # Keep the original 1,000-sentence dev/test convention, with a reproducible split.
shuffled = pd.Series(range(len(sentences))).sample(frac=1, random_state=42).tolist()
test_indices = shuffled[:1_000]
dev_indices = shuffled[1_000:2_000]
train_indices = shuffled[2_000:]

train = [sentences[index] for index in train_indices]
dev = [sentences[index] for index in dev_indices]
test = [sentences[index] for index in test_indices]

print(f"Loaded {len(sentences):,} sentences from {data_path}")
print(f"Train: {len(train):,} | Dev: {len(dev):,} | Test: {len(test):,}")

Loaded 100,000 sentences from e:\STUDY\NLP\Lab 1\outputs\tokenized_sentences_with_source.txt
Train: 98,000 | Dev: 1,000 | Test: 1,000


In [35]:
class NGramLanguageModel:
    """Unsmoothed maximum-likelihood language model for any n-gram order."""

    START = "<s>"
    END = "</s>"

    def __init__(self, n):
        if not isinstance(n, int) or n < 1:
            raise ValueError("n must be a positive integer")
        self.n = n
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocabulary = set()
        self.is_fitted = False

    def _padded(self, sentence):
        tokens = list(sentence)
        return [self.START] * (self.n - 1) + tokens + [self.END] # add <s>..</s>

    def fit(self, corpus):
        """Count n-grams and their contexts from an iterable of token lists."""
        self.ngram_counts.clear()
        self.context_counts.clear()
        self.vocabulary.clear()

        for sentence in corpus:
            padded = self._padded(sentence)
            self.vocabulary.update(padded)
            for index in range(len(padded) - self.n + 1):
                ngram = tuple(padded[index:index + self.n])
                context = ngram[:-1]  # for trigram denominator is bigram of history
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1

        self.is_fitted = True
        return self

    def probability(self, ngram):
        """Return P(last token | preceding n-1 tokens), or zero if unseen."""
        if not self.is_fitted:
            raise RuntimeError("Fit the model before requesting probabilities")
        ngram = tuple(ngram)
        if len(ngram) != self.n:
            raise ValueError(f"Expected an n-gram of length {self.n}")
        context = ngram[:-1]
        denominator = self.context_counts[context]
        return self.ngram_counts[ngram] / denominator if denominator else 0.0

    def predict_next(self, context, top_k=5):
        """Return the most likely next tokens for a context."""
        if not self.is_fitted:
            raise RuntimeError("Fit the model before making predictions")
        context = tuple(context)[-(self.n - 1):] if self.n > 1 else ()
        candidates = [
            (ngram[-1], count / self.context_counts[ngram[:-1]])
            for ngram, count in self.ngram_counts.items()
            if ngram[:-1] == context
        ]
        return sorted(candidates, key=lambda item: (-item[1], item[0]))[:top_k]

    def sentence_log_probability(self, sentence):
        """Return the log probability; unseen n-grams produce negative infinity."""
        total = 0.0
        padded = self._padded(sentence)
        for index in range(len(padded) - self.n + 1):
            probability = self.probability(padded[index:index + self.n])
            if probability == 0.0:
                return float("-inf")
            total += log(probability)
        return total

    def perplexity(self, corpus):
        """Compute token perplexity, returning infinity for unseen n-grams."""
        log_probability = 0.0
        token_count = 0
        for sentence in corpus:
            log_probability += self.sentence_log_probability(sentence)
            if log_probability == float("-inf"):
                return float("inf")
            token_count += len(sentence) + 1  # Include the end-of-sentence token.
        return exp(-log_probability / token_count) if token_count else float("inf")

    def __repr__(self):
        return f"NGramLanguageModel(n={self.n})"

## Train the four models

In [36]:
models = {
    "Unigram": NGramLanguageModel(1),
    "Bigram": NGramLanguageModel(2),
    "Trigram": NGramLanguageModel(3),
    "Quadrigram": NGramLanguageModel(4),
}

for model in models.values():
    model.fit(train)
    print(f"{model}: {len(model.ngram_counts):,} unique n-grams")

NGramLanguageModel(n=1): 139,533 unique n-grams
NGramLanguageModel(n=2): 847,597 unique n-grams
NGramLanguageModel(n=3): 1,249,480 unique n-grams
NGramLanguageModel(n=4): 1,389,285 unique n-grams


## Evaluation and next-token prediction

The models are intentionally unsmoothed, so a held-out sentence containing an unseen n-gram has infinite perplexity. This makes the limitation of maximum-likelihood estimates explicit.

In [37]:
evaluation = []
for name, model in models.items():
    evaluation.append({
        "model": name,
        "order": model.n,
        "unique_ngrams": len(model.ngram_counts),
        "dev_perplexity": model.perplexity(dev),
        "test_perplexity": model.perplexity(test),
    })

results = pd.DataFrame(evaluation)
display(results)

example_context = train[0][:3]
print(f"Example context: {example_context}")
for name, model in models.items():
    print(f"{name:10} -> {model.predict_next(example_context, top_k=3)}")

,model,order,unique_ngrams,dev_perplexity,test_perplexity
0,Unigram,1,139533,inf,inf
1,Bigram,2,847597,inf,inf
2,Trigram,3,1249480,inf,inf
3,Quadrigram,4,1389285,inf,inf


Example context: ['અને', 'હું', 'માનું']
Unigram    -> [('</s>', 0.06271217352074461), ('છે', 0.03959249885614275), ('અને', 0.017881928335343748)]
Bigram     -> [('છું', 0.8823529411764706), ('</s>', 0.058823529411764705), ("છું'", 0.058823529411764705)]
Trigram    -> [('છું', 1.0)]
Quadrigram -> [('છું', 1.0)]


## Add-one (Laplace) smoothing

For every n-gram order, the smoothed conditional probability is:

$$P_{Laplace}(w_n \mid w_1,\ldots,w_{n-1}) = \frac{C(w_1,\ldots,w_n)+1}{C(w_1,\ldots,w_{n-1})+V}$$

Here `V` is the training vocabulary size, so unseen n-grams receive a non-zero probability.

In [38]:
class LaplaceNGramLanguageModel(NGramLanguageModel):
    """N-gram model using add-one/Laplace smoothing."""

    def probability(self, ngram):
        if not self.is_fitted:
            raise RuntimeError("Fit the model before requesting probabilities")
        ngram = tuple(ngram)
        if len(ngram) != self.n:
            raise ValueError(f"Expected an n-gram of length {self.n}")

        context = ngram[:-1]
        vocabulary_size = len(self.vocabulary)
        return (self.ngram_counts[ngram] + 1) / (
            self.context_counts[context] + vocabulary_size
        )

    def predict_next(self, context, top_k=5):
        """Return the most likely next tokens under the smoothed model."""
        if not self.is_fitted:
            raise RuntimeError("Fit the model before making predictions")
        context = tuple(context)[-(self.n - 1):] if self.n > 1 else ()
        candidates = [
            (token, self.probability(context + (token,)))
            for token in self.vocabulary
            if token != self.START
        ]
        return sorted(candidates, key=lambda item: (-item[1], item[0]))[:top_k]

laplace_models = {
    "Laplace Unigram": LaplaceNGramLanguageModel(1),
    "Laplace Bigram": LaplaceNGramLanguageModel(2),
    "Laplace Trigram": LaplaceNGramLanguageModel(3),
    "Laplace Quadrigram": LaplaceNGramLanguageModel(4),
}

for model in laplace_models.values():
    model.fit(train)
    print(f"{model}: V={len(model.vocabulary):,}")

NGramLanguageModel(n=1): V=139,533
NGramLanguageModel(n=2): V=139,534
NGramLanguageModel(n=3): V=139,534
NGramLanguageModel(n=4): V=139,534


In [39]:
laplace_evaluation = []
for name, model in laplace_models.items():
    laplace_evaluation.append({
        "model": name,
        "order": model.n,
        "vocabulary_size": len(model.vocabulary),
        "dev_perplexity": model.perplexity(dev),
        "test_perplexity": model.perplexity(test),
    })

laplace_results = pd.DataFrame(laplace_evaluation)
display(laplace_results)
print(f"Example context: {example_context}")
for name, model in laplace_models.items():
    print(f"{name:20} -> {model.predict_next(example_context, top_k=3)}")

,model,order,vocabulary_size,dev_perplexity,test_perplexity
0,Laplace Unigram,1,139533,5296.248616,5387.214799
1,Laplace Bigram,2,139534,22819.809105,22697.418062
2,Laplace Trigram,3,139534,70923.914029,71383.797726
3,Laplace Quadrigram,4,139534,97006.699929,98420.187478


Example context: ['અને', 'હું', 'માનું']
Laplace Unigram      -> [('</s>', 0.05757219361918615), ('છે', 0.036347657305601834), ('અને', 0.01641671973437166)]
Laplace Bigram       -> [('છું', 0.000114653424196172), ('</s>', 1.43316780245215e-05), ("છું'", 1.43316780245215e-05)]
Laplace Trigram      -> [('છું', 6.449671066775594e-05), ("'", 7.166301185306216e-06), ("''", 7.166301185306216e-06)]
Laplace Quadrigram   -> [('છું', 1.4333321388898843e-05), ("'", 7.1666606944494216e-06), ("''", 7.1666606944494216e-06)]
